# 11 Production Agent Architectures: Coding Assistant Sandbox Simulator

**Scenario**: Implement a simulated system data flow for an AI Coding Assistant executing code compilation checks queryable via local Ollama `llama3.1:latest`.

## Step 1: Ollama Interface Setup

In [1]:
import urllib.request
import json

def query_ollama(prompt, model="llama3.1:latest"):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req) as response:
            res = json.loads(response.read().decode("utf-8"))
            return res.get("response", "").strip()
    except Exception as e:
        return "def sum_values(a, b):\n    return a + b"

print("System Warmup:", query_ollama("Say code in one word."))

System Warmup: Python.


## Step 2: Implement Code Sandbox Compiler Simulator

In [2]:
class CodingAssistantSimulator:
    def __init__(self):
        self.sandbox_files = {"main.py": ""}
        
    def run_sandbox_compiler(self):
        code = self.sandbox_files["main.py"]
        print(f"  [SANDBOX] Compiling main.py...\n  Code contents:\n{code}")
        try:
            # Sandbox compiler execution check
            exec_scope = {}
            exec(code, exec_scope)
            result = exec_scope["run"]()
            print(f"  [SANDBOX] Output: SUCCESS | Result: {result}")
            return f"SUCCESS: {result}"
        except Exception as e:
            print(f"  [SANDBOX] Output: FAILED | Error: {e}")
            return f"COMPILE ERROR: {e}"
            
    def solve_bug(self, bug_description):
        print(f"[AGENT] Goal: '{bug_description}'")
        
        # Iteration 1: Attempt to write code
        prompt = f"""Goal: {bug_description}
Write a Python function named 'run()' that returns the sum of 15 and 35.
Respond ONLY with the raw python function block."""
        
        code_solution = query_ollama(prompt)
        # Clean any markdown fences
        if "```" in code_solution:
            lines = code_solution.split("\n")
            code_solution = "\n".join([l for l in lines if not l.startswith("```")])
            
        self.sandbox_files["main.py"] = code_solution.strip()
        
        # 2. RUN SANDBOX
        status = self.run_sandbox_compiler()
        
        # 3. SELF-CORRECT / REPLAN IF ERROR OCCURS
        if "COMPILE ERROR" in status:
            print("\n[AGENT] Compile failed. Requesting self-correction update...")
            prompt_correction = f"""Fix this compilation error.
Error: {status}
Code:\n{self.sandbox_files['main.py']}
Respond ONLY with the corrected raw python function named 'run()'."""
            corrected_code = query_ollama(prompt_correction)
            self.sandbox_files["main.py"] = corrected_code.strip()
            status = self.run_sandbox_compiler()
            
        return status

assistant = CodingAssistantSimulator()
final_status = assistant.solve_bug("Implement sum helper yielding 15 + 35")
print(f"\nFinal Run Status: {final_status}")

[AGENT] Goal: 'Implement sum helper yielding 15 + 35'


  [SANDBOX] Compiling main.py...
  Code contents:
def run():
    def sum_helper(a, b):
        return a + b
    
    result = sum_helper(15, 35)
    return result
  [SANDBOX] Output: SUCCESS | Result: 50

Final Run Status: SUCCESS: 50


## Output Explanation & Complexity Analysis

- **Code Sandbox Compiler**: Simulates compiling the generated code block using standard python `exec()`. If compilation fails, it triggers the LLM feedback channel to obtain corrected functions.
- **Time Complexity**: $O(T \cdot (t_{\text{compile}} + t_{\text{inference}}))$ total latency.
- **Space Complexity**: $O(N_c)$ memory scale representation.
- **Component Denotations**:
  - $T$: Number of repair loops ($T \le 2$ runs).
  - $t_{\text{compile}}$: Physical compiler execution time.
  - $t_{\text{inference}}$: LLM query latency.
  - $N_c$: Code size in bytes.